# Notebook 1: Cut the Kade pages into lines

The Kurrent adapter we build on was trained on pictures of **single lines** of handwriting, so it
reads one line at a time. This notebook gets the Kade page images, cuts every page into one image
per line with **kraken**, and pairs each line image with its line of the correct transcription,
ready for training in Notebook 2.

You run this once for the whole project. After that the line images are saved on this instance
(and, if you do the last step, in S3 for the rest of the team).

**Time:** about 10 minutes of setup, then roughly 45 minutes for kraken to cut all 436 pages.
Run the cells from top to bottom with **Shift + Enter** (the same on Mac and Windows).
Kernel: **conda_pytorch_p310** (or whichever `conda_pytorch` kernel your instance lists).

## Step 1. Set up the folders

This cell decides where everything goes. You don't need to change anything in it.

In [ ]:
import os
import sys

# Everything this project downloads or creates lives here. It is on the
# notebook instance's storage volume, so it survives Stop and Start.
WORK = "/home/ec2-user/SageMaker/badger-finetune"
DATA = os.path.join(WORK, "data")            # page images, labels, line images
RUNS = os.path.join(WORK, "runs")            # one folder per training run
SCRIPTS = os.path.abspath("scripts")         # the scripts folder next to this notebook
for folder in (WORK, DATA, RUNS):
    os.makedirs(folder, exist_ok=True)
sys.path.insert(0, SCRIPTS)

# Keep the 9 GB model download on the big volume, not the small system disk.
os.environ["HF_HOME"] = os.path.join(WORK, "hf_cache")

SHARED_BUCKET = "ml-marathon-shared"   # the class bucket with the Badger Scribe data
SHARED_PREFIX = "badger-scribe"
print("Working folder:", WORK)

## Step 2. Install kraken (only the first time)

kraken needs a newer version of PyTorch than the one this notebook's kernel uses for training. If
we installed it here, it would replace that version and break Notebook 2. So kraken gets its own,
separate Python environment inside the working folder. We never switch to it: when it's time to
cut pages, Notebook 1 runs the kraken script with that environment's Python.

The environment lives on the instance's storage volume, so this cell only does real work the
first time (about 3 minutes). After that it says the environment is ready and finishes in seconds.

In [ ]:
import subprocess

KRAKEN_ENV = os.path.join(WORK, "kraken-env")
KRAKEN_PYTHON = os.path.join(KRAKEN_ENV, "bin", "python")

# uv downloads Python and packages. Point its downloads at the working folder,
# so they survive a Stop too.
os.environ["UV_PYTHON_INSTALL_DIR"] = os.path.join(WORK, "uv-python")
os.environ["UV_CACHE_DIR"] = os.path.join(WORK, "uv-cache")

def kraken_works():
    test = "import kraken, importlib.metadata as m; print('kraken', m.version('kraken'))"
    return os.path.exists(KRAKEN_PYTHON) and subprocess.run([KRAKEN_PYTHON, "-c", test]).returncode == 0

if kraken_works():
    print("The kraken environment is ready.")
else:
    print("Setting up the kraken environment. This takes about 3 minutes ...")
    !{sys.executable} -m pip install -q uv
    !{sys.executable} -m uv venv -q --python 3.12 {KRAKEN_ENV}
    !{sys.executable} -m uv pip install -q --python {KRAKEN_PYTHON} --torch-backend cpu kraken
    if not kraken_works():
        raise SystemExit("kraken didn't install. Look at the messages above, or see Troubleshooting in README.md.")
    print("The kraken environment is ready.")

## Step 3. Download the page images and labels

The class bucket holds the Kade page images and the two lists of pages: `train.csv` (pages with
correct transcriptions) and `test.csv` (pages Kaggle scores us on). This copies them to the
instance. Pages already downloaded are skipped, so it's safe to run again.

If you get an *Access Denied* error, your notebook's role can't read the class bucket; see
Troubleshooting in README.md.

In [ ]:
import boto3

s3 = boto3.client("s3")
for name in ("train.csv", "test.csv"):
    s3.download_file(SHARED_BUCKET, f"{SHARED_PREFIX}/{name}", os.path.join(DATA, name))

images_dir = os.path.join(DATA, "images")
os.makedirs(images_dir, exist_ok=True)
pages = s3.get_paginator("list_objects_v2").paginate(Bucket=SHARED_BUCKET, Prefix=f"{SHARED_PREFIX}/images/kade_")
keys = [obj["Key"] for page in pages for obj in page.get("Contents", []) if obj["Key"].endswith(".jpg")]
new = 0
for key in keys:
    path = os.path.join(images_dir, os.path.basename(key))
    if not os.path.exists(path):
        s3.download_file(SHARED_BUCKET, key, path)
        new += 1
print(f"{len(keys)} Kade page images in the bucket, {new} downloaded now, all in {images_dir}")

## Step 4. Cut the pages into lines

kraken looks at each page, finds every line of writing, and saves it as its own image in
`data/kade_lines/<page name>/000.png, 001.png, ...`, from top to bottom.

**Start small.** `LIMIT = 5` cuts just five pages, so you can check the result in Step 5 first.
Then set `LIMIT = None` and run this cell again for all of them. Pages that are already done are
skipped, so nothing is repeated, and if it stops halfway you can simply run it again.

It works on several pages at a time, one per processor. Expect about 6 seconds per page on a
4-processor instance: around 45 minutes for all 436. The last number on each line is the time left.

In [ ]:
LIMIT = 5   # then None for every page

lines_dir = os.path.join(DATA, "kade_lines")
limit = f"--limit {LIMIT}" if LIMIT else ""
!{KRAKEN_PYTHON} {SCRIPTS}/segment_lines.py --images {images_dir} --out {lines_dir} {limit}

## Step 5. Check the result by eye

This shows one page next to the lines kraken cut out of it. Each line image should hold exactly one
whole line of writing: not two lines stuck together, not half a line. A few mistakes are fine
(Step 6 leaves those pages out of training), but if most pages look wrong, tell the team before
going on.

In [ ]:
from IPython.display import display
from PIL import Image

done = sorted(d for d in os.listdir(lines_dir) if not d.endswith(".partial"))
page = done[0]   # or type a page name, e.g. "kade_112_p004"
print(page)
img = Image.open(os.path.join(images_dir, page + ".jpg"))
img.thumbnail((500, 800))
display(img)
for name in sorted(os.listdir(os.path.join(lines_dir, page))):
    print(name)
    display(Image.open(os.path.join(lines_dir, page, name)))

## Step 6. Pair the lines with the transcriptions

For training we need to know which line of the transcription belongs to which line image. When
kraken found **exactly as many lines** on a page as its transcription has, we pair them in order:
image 1 with line 1, image 2 with line 2, and so on. Pages where the numbers differ are left out of
training, since we can't be sure which line is which.

Two things are taken out of the transcriptions first: blank lines, and the `Postmark ...` line
that some postcards have (the postmark is on the back of the card, so it isn't in the image).

The 11 pages of letter `kade_112` are the team's **dev set**: they're never trained on, only used
to score the model.

**Read the first line it prints.** If well under 70% of the training pages are usable, something
is off with the line cutting; look at a few of the pages listed below it.

In [ ]:
import pandas as pd
import prepare_training_data

counts = prepare_training_data.build(DATA, os.path.join(DATA, "train.csv"))
bad = counts[(~counts["match"]) & (counts["kraken_lines"] > 0)]
if len(bad):
    print("\nSome pages whose line counts don't match (kraken vs. transcription):")
    print(bad.head(10)[["page_id", "kraken_lines", "label_lines"]].to_string(index=False))
else:
    print("\nEvery segmented page matches its transcription.")

## Step 7. Save the line images to S3 for the team (optional)

The line images are on this instance only. Copy them to your own bucket so teammates (or you, on
a new instance) can skip this notebook. Fill in your bucket's name first.

In [ ]:
MY_BUCKET = "your-bucket-name"   # <-- your own bucket

!aws s3 sync {DATA} s3://{MY_BUCKET}/badger-finetune/data --exclude "images/*" --quiet
print(f"Saved to s3://{MY_BUCKET}/badger-finetune/data")

**Done.** Go on to **Notebook 2** (`2_finetune.ipynb`).

To use someone else's line images instead of running this notebook, run Step 1, then in a cell:
`!aws s3 sync s3://<their bucket>/badger-finetune/data {DATA}`